In [ ]:
!pip install -q -U vllm transformers
print("설치 완료")

In [ ]:
# ===== CONFIG — 여기만 수정 =====
PIPELINE   = "dpo"        # "dpo"(확정 0.734) | "ensemble"(dpo+gemini, ~2배 느림)
TEST_CSV = "/kaggle/input/datasets/h70jun/test-real/test_submission.csv"
OUT_CSV    = "/kaggle/working/submission.csv"

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
DPO_LORA   = "/kaggle/input/datasets/h70jun/qlora-dpo/qlora_dpo"
GEM_LORA   = "/kaggle/input/datasets/h70jun/deeplearning2/qlora_gemini2_backup"
VER_LORA   = "/kaggle/input/datasets/h70jun/deeplearning2/qlora_verifier_backup"

N_TOTAL       = 32        # 여유되면 64 
LAMBDA        = 0.7       # Score = SC + LAMBDA * P(Yes) * N_TOTAL
CHUNK         = 50
MAX_INPUT_TOK = 2900
SEED          = 42
SAMPLE_N      = 0         

SYSTEM_PROMPT = ("You are a careful math problem solver. Solve the problem step by step. "
    "The final answer is always an integer. End your response with the final answer in the format \\boxed{answer}.")
VERIFY_SYSTEM = ("You are a math solution verifier. Given a problem and a proposed solution, "
    "judge whether the solution's final answer is correct. Respond with exactly 'Yes' or 'No'.")
YES_STRINGS = {"Yes", " Yes", "yes", " yes", "YES", " YES"}
ID_COL, ANSWER_COL = "id", "answer"
print("CONFIG:", PIPELINE, "| N_TOTAL", N_TOTAL, "| SAMPLE_N", SAMPLE_N)

In [ ]:
# ===== 경로 확인 =====
import os
need = {"DPO": DPO_LORA, "verifier": VER_LORA}
if PIPELINE == "ensemble":
    need["Gemini"] = GEM_LORA
ok = True
for name, p in need.items():
    e = os.path.exists(f"{p}/adapter_model.safetensors")
    print(("OK " if e else "XX "), name, p)
    ok = ok and e
te = os.path.exists(TEST_CSV)
print(("OK " if te else "XX "), "test", TEST_CSV)
assert ok and te, "빠진 Input 있음 -> Add Input 하고 다시 실행"
print("전부 준비됨 — 진행 가능")

In [ ]:
# ===== 로드 + 헬퍼 =====
import re, math, time
from collections import Counter, defaultdict
import pandas as pd
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

N_PER, MAX_LORAS = (N_TOTAL // 2, 3) if PIPELINE == "ensemble" else (N_TOTAL, 2)

def extract_answer(text):
    boxed = re.findall(r"\\boxed\{([^}]*)\}", text)
    cand = boxed[-1] if boxed else None
    if cand is None:
        nums = re.findall(r"-?\d[\d,]*", text)
        cand = nums[-1] if nums else None
    if cand is None:
        return None
    try:
        v = int(cand.replace(",", ""))
        return None if abs(v) > 10**18 else v
    except ValueError:
        return None

def p_yes(d):
    s = 0.0
    for lp in d.values():
        if lp.decoded_token in YES_STRINGS:
            s += math.exp(lp.logprob)
    return min(s, 1.0)

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
llm = LLM(model=BASE_MODEL, dtype="float16", gpu_memory_utilization=0.85,
          max_model_len=3072, enforce_eager=True, enable_lora=True,
          max_lora_rank=16, max_loras=MAX_LORAS, tensor_parallel_size=2)
DPO_REQ = LoRARequest("dpo", 1, DPO_LORA)
GEM_REQ = LoRARequest("gemini", 2, GEM_LORA)
VER_REQ = LoRARequest("ver", 3, VER_LORA)
GEN_LORAS = [DPO_REQ] if PIPELINE == "dpo" else [DPO_REQ, GEM_REQ]
gen_params = SamplingParams(temperature=0.8, top_p=0.95, n=N_PER, max_tokens=512, seed=SEED)
ver_params = SamplingParams(temperature=0.0, max_tokens=1, logprobs=20)
print("로드 완료:", PIPELINE, "| max_loras", MAX_LORAS, "| N_PER", N_PER)

In [ ]:
# ===== 추론 (블록 단위 생성->채점->선택, 블록마다 저장 + ETA) =====
lb = pd.read_csv(TEST_CSV)
lb.columns = [c.strip() for c in lb.columns]
print("columns:", list(lb.columns))
id_col = ID_COL     if ID_COL     in lb.columns else lb.columns[0]
q_col  = "question" if "question" in lb.columns else lb.columns[1]
if SAMPLE_N:
    lb = lb.head(SAMPLE_N)
    print(f"*** DRY RUN {SAMPLE_N}문제 — ETA만 보고 제출하지 말 것 ***")
ids       = lb[id_col].tolist()
questions = lb[q_col].tolist()
print("questions:", len(questions))

def solve_block(qs):
    prompts = [tok.apply_chat_template(
        [{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":q}],
        tokenize=False, add_generation_prompt=True) for q in qs]
    counter = [Counter() for _ in qs]
    reps    = [defaultdict(list) for _ in qs]
    for lora in GEN_LORAS:
        for j, out in enumerate(llm.generate(prompts, gen_params, lora_request=lora)):
            for o in out.outputs:
                a = extract_answer(o.text)
                if a is None:
                    continue
                counter[j][a] += 1
                if len(reps[j][a]) < 2:
                    reps[j][a].append(o.text.strip())
    vprompts, vindex = [], []
    for j, q in enumerate(qs):
        for ans, sols in reps[j].items():
            for sol in sols:
                user = f"Problem:\n{q}\n\nProposed solution:\n{sol}\n\nIs the final answer correct?"
                p = tok.apply_chat_template(
                    [{"role":"system","content":VERIFY_SYSTEM},{"role":"user","content":user}],
                    tokenize=False, add_generation_prompt=True)
                pid = tok(p)["input_ids"]
                if len(pid) > MAX_INPUT_TOK:
                    p = tok.decode(pid[:MAX_INPUT_TOK])
                vprompts.append(p); vindex.append((j, ans))
    pyes = defaultdict(float)
    if vprompts:
        for (j, ans), o in zip(vindex, llm.generate(vprompts, ver_params, lora_request=VER_REQ)):
            pyes[(j, ans)] = max(pyes[(j, ans)], p_yes(o.outputs[0].logprobs[0]))
    out_ans = []
    for j in range(len(qs)):
        cnt = counter[j]
        if not cnt:
            out_ans.append(0); continue
        best_a, best_s = None, -1
        for a in cnt:
            s = cnt[a] + LAMBDA * pyes.get((j, a), 0.0) * N_TOTAL
            if s > best_s:
                best_s, best_a = s, a
        out_ans.append(int(best_a))
    return out_ans

final_ids, final_ans = [], []
t0 = time.time()
nb_ = (len(questions) + CHUNK - 1) // CHUNK
for b, s in enumerate(range(0, len(questions), CHUNK), 1):
    final_ans += solve_block(questions[s:s+CHUNK])
    final_ids += ids[s:s+CHUNK]
    amap = dict(zip(final_ids, final_ans))
    out_df = lb.copy()
    out_df[ANSWER_COL] = out_df[id_col].map(amap).fillna(0).astype("int64")
    out_df.to_csv(OUT_CSV, index=False)
    el = time.time() - t0
    print(f"block {b}/{nb_}  done {len(final_ids)}/{len(questions)}  {el/60:.1f}m  ETA {el/b*(nb_-b)/60:.1f}m")
print("wrote", OUT_CSV, len(final_ids), f"| total {(time.time()-t0)/60:.1f}m")